# フィジカルAI開発体験

## フィジカルAI

フィジカルAIは、AIを搭載したロボットが現実を知覚し、推論し、物理空間と相互作用しながら動作する技術

[HITTER: A Humanoid Table Tennis  Robot via Hierachical Planning and Learning][1]:

![](image/HITTER.png)

[1]: https://www.youtube.com/watch?v=tOfPKW6D3gE

## LeRobot

LeRobot（ル・ロボット）は、Hugging Faceが提供しているオープンソースのフィジカルAIライブラリ

学習データの作成、データセットの管理、シミュレーション、ロボットの訓練ができる

[LeRobot][1]:

![](image/LeRobot.png)

[1]: https://huggingface.co/lerobot

## SO101（エスオー・ワンオーワン）

SO101は、Hugging Faceが提供しているオープンソースのアームロボット

秋月電子で購入可能

- [SO-101 オープンソースアームキット][1] 39,980円
- [3Dプリントパーツ][2] 7,280円

![](image/SO101.png)

[1]: https://akizukidenshi.com/catalog/g/g131169/
[2]: https://akizukidenshi.com/catalog/g/g131222/

## ロボットの訓練

3つのトレンド

- 模倣学習モデル（Imitation Learning）
- VLA（Vision Language Action, 視覚言語行動モデル）
- 強化学習モデル（Reinforcement Learning）

仕組み、学習データ量、学習コスト、推論速度、環境（物理/シミュレーション）が異なる

## 模倣学習モデル

模倣学習モデルは、観測データからアクションを直接推定するモデル

カメラ画像・モーターの状態から未来のモーターの状態を直接予測する

汎用性が低いが、学習コストが低く、リアルタイムな推論が可能

[ACT][1]:

![](image/ALOHA.png)

[1]: https://tonyzhaozh.github.io/aloha/

## VLA

VLAは、VLM（Vision Language Model, 視覚言語モデル）にアクションを生成する機能を加えたモデル

プロンプト・カメラ画像・モーターの状態をVLMに入力し、未来のモーターの状態を予測する

リアルタイムの推論が難しいが、基盤モデルを再利用でき、汎用性が高い

[$\pi_{0.5}$][1]:

![](image/pi0-5.png)

[1]: https://www.physicalintelligence.company/blog/pi05

## 強化学習モデル

強化学習モデルは、物理シミュレーションでロボットを動かし、訓練したモデル

シミュレーションから実環境への適応が難しく学習コストが高いが、リアルタイムに推論でき高性能

[Isaac Sim][0]:

![](image/IsaacSim.png)

[0]: https://docs.isaacsim.omniverse.nvidia.com/5.1.0/robot_simulation/ext_isaacsim_robot_policy_example.html

## 模倣学習用のデータセットを作ろう

SO-101の場合、ACT（Action Chunking Transformer）という模倣学習モデルが最も再現性が高い

= 少ないデータ量で簡単なタスクを習得できる

## セットアップ

<img src="image/env.png" style="width: 600px" />

## 環境構築

- [macOS/Linux](https://docs.conda.io/projects/conda/en/latest/user-guide/install/macos.html#installing-in-silent-mode)
- [Windows](https://docs.conda.io/projects/conda/en/latest/user-guide/install/windows.html#installing-on-windows)

In [ ]:
# macOSかLinux
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-MacOSX-x86_64.sh -O ~/miniconda.sh
!bash ~/miniconda.sh -b -p $HOME/miniconda
!source $HOME/miniconda/bin/activate
!conda init --all
!conda create -y -n lerobot-v0.4.0 python=3.10
!conda activate lerobot-v0.4.0
!conda install ffmpeg -c conda-forge
if not os.path.exists("lerobot"):
    !git clone https://github.com/huggingface/lerobot.git
%pip install -qe lerobot
%pip install -qe "lerobot[feetech]" # モーターのドライバー

# Conda環境を設定する
!conda activate lerobot-v0.4.0

## USBポートの特定

In [ ]:
# 1. マイコンボードをUSBで接続する
# 2. find_portコマンドを実行
!lerobot-find-port

# 3. USBを外して、エンターキーを押す
# 4. 「/dev/tty.usbmodem5A7A0182121」のようなポートが出力されるので変数に保存する
!export LEADER_PORT="/dev/tty.usbmodem5A7A0182121"
!export FOLLOWER_PORT="/dev/tty.usbmodem5A7A0182121"

## キャリブレーション

In [ ]:
# IDを設定する
!export LEADER_ID="leader_arm_1"
!export FOLLOWER_ID="follower_arm_1"

# Followerのキャリブレーション
!lerobot-calibrate \
    --robot.type=so101_follower \
    --robot.port=$FOLLOWER_PORT \
    --robot.id=$FOLLOWER_ID

# Leaderのキャリブレーション
!lerobot-calibrate \
    --teleop.type=so101_leader \
    --teleop.port=$LEADER_PORT \
    --teleop.id=$LEADER_ID

フォロワーアームの中間位置:

<img src="image/follower.png" style="width: 400px" />

## テレオペレーション


In [ ]:
!lerobot-teleoperate \
    --robot.type=so101_follower \
    --robot.port=$FOLLOWER_PORT \
    --robot.id=$FOLLOWER_ID \
    --teleop.type=so101_leader \
    --teleop.port=$LEADER_PORT \
    --teleop.id=$LEADER_ID

## カメラの検出

In [ ]:
!lerobot-find-cameras opencv

!export CAMERAS="{ image.top.right: {type: opencv, index_or_path: 1, width: 640, height: 480, fps: 30}, image.wrist.left: {type: opencv, index_or_path: 0, width: 640, height: 480, fps: 30}}"


## カメラ付きテレオペレーション

In [ ]:
!lerobot-teleoperate \
    --robot.type=so101_follower \
    --robot.port=$FOLLOWER_PORT \
    --robot.id=$FOLLOWER_ID \
    --robot.cameras=$CAMERAS \
    --teleop.type=so101_leader \
    --teleop.port=$LEADER_PORT \
    --teleop.id=$LEADER_ID \
    --display_data=true

## データセットの記録

In [ ]:
!export HF_USER="EngineerCafeJP"
!export DATE=$(date +%Y-%m-%d-%H-%M-%S)
!export REPO_ID="${HF_USER}/record-test_${DATE}"
!export NUM_EPISODES=50
!export SINGLE_TASK="Grab the green bottle cap and place it in the brown box."
!export SINGLE_TASK="Put the green bottle cap on the blue bottle cap."

!lerobot-record \
    --robot.type=so101_follower \
    --robot.port=$FOLLOWER_PORT \
    --robot.id=$FOLLOWER_ID \
    --robot.cameras=$CAMERAS \
    --teleop.type=so101_leader \
    --teleop.port=$LEADER_PORT \
    --teleop.id=$LEADER_ID \
    --display_data=true \
    --dataset.repo_id=$REPO_ID \
    --dataset.single_task=$SINGLE_TASK \
    --dataset.num_episodes=$NUM_EPISODES \
    --resume=false

## データセットの可視化

[Visualize dataset][1]

![](image/preview.png)

[1]: https://huggingface.co/spaces/lerobot/visualize_dataset

## 訓練の準備

In [ ]:
import os

try:
    from google.colab import userdata
    HF_USER = userdata.get("HF_USER")
    HF_TOKEN = userdata.get("HF_TOKEN")
    WANDB_API_KEY = userdata.get("WANDB_API_KEY")

    !pip install -q condacolab
    import condacolab
    condacolab.install()
except:
    from dotenv import load_dotenv
    load_dotenv()
    HF_USER = os.getenv("HF_USER")
    HF_TOKEN = os.getenv("HF_TOKEN")
    WANDB_API_KEY = os.getenv("WANDB_API_KEY")

assert HF_USER is not None, "HF_USER is not set"
assert HF_TOKEN is not None, "HF_TOKEN is not set"
assert WANDB_API_KEY is not None, "WANDB_API_KEY is not set"

if not os.path.exists("lerobot"):
    !git clone https://github.com/huggingface/lerobot.git
    !conda install ffmpeg=7.1.1 -c conda-forge
    !cd lerobot && pip install -e .

import wandb
wandb.login(key=WANDB_API_KEY)

## 訓練

In [ ]:
# A100の場合はbatch_size=64

!export OUTPUT_DIR="outputs/train/act_so101_test"
!export JOB_NAME="act_so101_test"
!eport POLICY_NAME="act_so101_test_${DATE}"

!cd lerobot && python src/lerobot/scripts/lerobot_train.py \
    --dataset.repo_id=$REPO_ID \
    --policy.type=act \
    --output_dir=$OUTPUT_DIR \
    --job_name=act-test_2025-10-25-15-32-04 \
    --policy.device=cuda \
    --policy.push_to_hub=true \
    --policy.repo_id=$HF_USER/act-test_2025-10-25-15-32-04 \
    --wandb.enable=true \
    --num_workers=4 \
    --batch_size=64 \
    --steps=100_000 \
    --save_freq=1000 \
    --wandb.enable=true

## チェックポイントのアップロード

In [ ]:
!export CHECKPOINT_NAME="act_so101_test"

!huggingface-cli upload ${HF_USER}/${CHECKPOINT_NAME} \
${OUTPUT_DIR}/checkpoints/last/pretrained_model

## 訓練の再開

In [ ]:
!lerobot-train \
    --config_path=${OUTPUT_DIR}/checkpoints/last/pretrained_model/train_config.json \
    --resume=true

## ACTの仕組み

ACT（Action Chunking Transformer）はカメラ画像とモーターの状態から、未来のモーターの状態（アクション）をまとめて予測するモデル

[Transformer][1]アーキテクチャを採用:

- エンコーダー: 入力データを圧縮する
- デコーダー: 圧縮されたデータを、コンテキスト情報を参照して解凍
- 入力データ・コンテキスト情報・正解の出力データで構成されるデータセットで訓練

※ 厳密ではないので注意

[1]: https://arxiv.org/pdf/1706.03762

![](image/transformer_Medium.png)

## ACTでのTransformer

既存のTransformerと同じ:

- 入力データ: 4枚のカメラ画像・現在のモーターの状態・スタイル変数
- コンテキストデータ: モーターの位置
- 出力データ: 未来のモーターの状態のシーケンス（アクション）

![](image/act-2.png)

## 訓練の結果

![](image/trained.png)

## 推論の実行

In [ ]:
!export EVAL_NAME="eval-so101-test_$(date +%Y-%m-%d-%H-%M-%S)"

!lerobot-record  \
    --robot.type=so101_follower \
    --robot.port=$FOLLOWER_PORT \
    --robot.cameras=$CAMERAS \
    --robot.id=$FOLLOWER_ID \
    --display_data=false \
    --dataset.repo_id=${HF_USER}/${EVAL_NAME} \
    --dataset.single_task=${SINGLE_TASK} \
    --policy.path=${HF_USER}/${POLICY_NAME}